# 🎓 Student & Admission Insights Dashboard
### A 360° view of enrollment patterns, admission trends, and course performance.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# ── Color palette ──────────────────────────────────────────────
PALETTE = {
    "primary":   "#2563EB",
    "secondary": "#10B981",
    "accent":    "#F59E0B",
    "danger":    "#EF4444",
    "gray":      "#6B7280",
    "light":     "#F3F4F6",
    "courses": px.colors.qualitative.Pastel,
}

# ── Chart helper ───────────────────────────────────────────────
def clean_layout(fig, title='', height=400):
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color='#111827'), x=0),
        height=height,
        margin=dict(l=20, r=20, t=50, b=20),
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family='Arial', size=12, color='#374151'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)',
                    font=dict(size=11)),
        xaxis=dict(showgrid=False, linecolor='#E2E8F0', tickfont=dict(size=11)),
        yaxis=dict(gridcolor='#F3F4F6', linecolor='#E2E8F0', tickfont=dict(size=11)),
    )
    return fig

print('✅ Libraries loaded.')

## 📂 1. Load & Prepare Data

In [ ]:
students_raw   = pd.read_csv('students.csv')
admissions_raw = pd.read_csv('admission.csv')

# Parse dates
students_raw['enrollment_date']  = pd.to_datetime(students_raw['enrollment_date'],  dayfirst=False, errors='coerce')
admissions_raw['admission_date'] = pd.to_datetime(admissions_raw['admission_date'], dayfirst=False, errors='coerce')

# Derived columns
students_raw['enrollment_month']   = students_raw['enrollment_date'].dt.to_period('M').astype(str)
admissions_raw['admission_month']  = admissions_raw['admission_date'].dt.to_period('M').astype(str)
admissions_raw['is_admitted']      = admissions_raw['status'] == 'Admitted'

# Use full datasets (no sidebar filters in notebook — filter below if needed)
students   = students_raw.copy()
admissions = admissions_raw.copy()

print(f'Students:   {len(students)} rows  |  Columns: {list(students.columns)}')
print(f'Admissions: {len(admissions)} rows  |  Columns: {list(admissions.columns)}')

## 🎛️ 2. Optional Filters
> Adjust values below and re-run from this cell to filter the entire notebook.

In [ ]:
# ── Edit these lists to filter; leave empty list [] to include ALL ──────
FILTER_COURSES   = []   # e.g. ['Data Science with Python', 'DevOps']
FILTER_MODES     = []   # e.g. ['Online']
FILTER_EDU       = []   # e.g. ['UG', 'PG']
FILTER_LOCATIONS = []   # e.g. ['Kochi', 'Calicut']

def apply(df, col, vals):
    return df[df[col].isin(vals)] if vals else df

students   = apply(apply(apply(students_raw,   'course', FILTER_COURSES), 'mode', FILTER_MODES), 'education_level', FILTER_EDU)
admissions = apply(apply(apply(admissions_raw, 'course', FILTER_COURSES), 'mode', FILTER_MODES), 'location', FILTER_LOCATIONS)

print(f'Filtered → Students: {len(students)}  |  Admissions: {len(admissions)}')

## 📊 3. KPI Summary

In [ ]:
total_students   = len(students)
total_admissions = len(admissions)
admitted         = admissions['is_admitted'].sum()
not_admitted     = (~admissions['is_admitted']).sum()
admission_rate   = round(admitted / total_admissions * 100, 1) if total_admissions else 0
internship_pct   = round((students['internship'] != 'No').sum() / total_students * 100, 1) if total_students else 0

kpi_html = f'''
<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:Arial">
  <div style="flex:1;min-width:160px;background:#fff;border:1px solid #E2E8F0;border-radius:12px;padding:20px;box-shadow:0 1px 4px rgba(0,0,0,.06)">
    <div style="font-size:11px;font-weight:700;color:#6B7280;text-transform:uppercase;letter-spacing:.6px">Total Students</div>
    <div style="font-size:34px;font-weight:800;color:#111827;line-height:1.2">{total_students}</div>
    <div style="font-size:12px;color:#9CA3AF">{len(students_raw)} in full dataset</div>
  </div>
  <div style="flex:1;min-width:160px;background:#fff;border:1px solid #E2E8F0;border-radius:12px;padding:20px;box-shadow:0 1px 4px rgba(0,0,0,.06)">
    <div style="font-size:11px;font-weight:700;color:#6B7280;text-transform:uppercase;letter-spacing:.6px">Total Inquiries</div>
    <div style="font-size:34px;font-weight:800;color:#111827;line-height:1.2">{total_admissions}</div>
    <div style="font-size:12px;color:#9CA3AF">{len(admissions_raw)} in full dataset</div>
  </div>
  <div style="flex:1;min-width:160px;background:#fff;border:1px solid #E2E8F0;border-radius:12px;padding:20px;box-shadow:0 1px 4px rgba(0,0,0,.06)">
    <div style="font-size:11px;font-weight:700;color:#6B7280;text-transform:uppercase;letter-spacing:.6px">Admission Rate</div>
    <div style="font-size:34px;font-weight:800;color:#2563EB;line-height:1.2">{admission_rate}%</div>
    <div style="font-size:12px"><span style="color:#10B981;font-weight:600">✔ {admitted} admitted</span> &nbsp;|&nbsp; <span style="color:#EF4444;font-weight:600">✘ {not_admitted} not</span></div>
  </div>
  <div style="flex:1;min-width:160px;background:#fff;border:1px solid #E2E8F0;border-radius:12px;padding:20px;box-shadow:0 1px 4px rgba(0,0,0,.06)">
    <div style="font-size:11px;font-weight:700;color:#6B7280;text-transform:uppercase;letter-spacing:.6px">Internship Rate</div>
    <div style="font-size:34px;font-weight:800;color:#10B981;line-height:1.2">{internship_pct}%</div>
    <div style="font-size:12px;color:#9CA3AF">Students with placement</div>
  </div>
</div>
'''
display(HTML(kpi_html))

## 👩‍🎓 4. Student Insights

In [ ]:
course_counts = students['course'].value_counts().reset_index()
course_counts.columns = ['Course', 'Students']
fig = px.bar(course_counts, x='Students', y='Course', orientation='h',
             color='Course', color_discrete_sequence=PALETTE['courses'])
fig.update_traces(marker_line_width=0)
fig.update_layout(showlegend=False)
clean_layout(fig, 'Students by Course', height=380)
fig.show()

In [ ]:
edu_counts = students['education_level'].value_counts().reset_index()
edu_counts.columns = ['Education Level', 'Count']
fig = px.pie(edu_counts, names='Education Level', values='Count', hole=0.55,
             color_discrete_sequence=[PALETTE['primary'], PALETTE['secondary'], PALETTE['accent'], '#A78BFA'])
fig.update_traces(textposition='outside', textinfo='percent+label',
                  marker=dict(line=dict(color='white', width=2)))
clean_layout(fig, 'Education Level Distribution')
fig.show()

In [ ]:
mode_counts = students['mode'].value_counts().reset_index()
mode_counts.columns = ['Mode', 'Count']
fig = px.bar(mode_counts, x='Mode', y='Count', color='Mode',
             color_discrete_map={'Online': PALETTE['primary'], 'Offline': PALETTE['secondary']},
             text='Count')
fig.update_traces(marker_line_width=0, textposition='outside')
fig.update_layout(showlegend=False)
clean_layout(fig, 'Online vs Offline Enrollment')
fig.show()

In [ ]:
fig = px.histogram(students, x='age', nbins=15,
                   color_discrete_sequence=[PALETTE['primary']])
fig.update_traces(marker_line_color='white', marker_line_width=1)
clean_layout(fig, 'Age Distribution of Students')
fig.show()

In [ ]:
enroll_trend = (students.groupby('enrollment_month').size()
                .reset_index(name='Count').sort_values('enrollment_month'))
fig = px.area(enroll_trend, x='enrollment_month', y='Count',
              color_discrete_sequence=[PALETTE['primary']], markers=True)
fig.update_traces(line_width=2.5, marker_size=5)
clean_layout(fig, 'Enrollment Trend Over Time')
fig.update_xaxes(tickangle=-30)
fig.show()

## 📋 5. Admission Insights

In [ ]:
adm_trend = (admissions.groupby('admission_month').size()
             .reset_index(name='Inquiries').sort_values('admission_month'))
fig = px.line(adm_trend, x='admission_month', y='Inquiries',
              color_discrete_sequence=[PALETTE['secondary']], markers=True)
fig.update_traces(line_width=2.5, marker_size=6)
clean_layout(fig, 'Admission Inquiries Over Time')
fig.update_xaxes(tickangle=-30)
fig.show()

In [ ]:
status_counts = admissions['status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']
fig = px.pie(status_counts, names='Status', values='Count', hole=0.55,
             color_discrete_map={'Admitted': PALETTE['secondary'], 'Not Admitted': PALETTE['danger']})
fig.update_traces(textposition='outside', textinfo='percent+label',
                  marker=dict(line=dict(color='white', width=2)))
clean_layout(fig, 'Admission Status Split')
fig.show()

In [ ]:
loc_counts = admissions['location'].value_counts().reset_index()
loc_counts.columns = ['Location', 'Count']
fig = px.bar(loc_counts, x='Location', y='Count', color='Location',
             color_discrete_sequence=PALETTE['courses'], text='Count')
fig.update_traces(marker_line_width=0, textposition='outside')
fig.update_layout(showlegend=False)
clean_layout(fig, 'Inquiries by Location')
fig.show()

In [ ]:
course_adm = admissions.groupby('course')['is_admitted'].agg(['sum', 'count']).reset_index()
course_adm.columns = ['Course', 'Admitted', 'Total']
course_adm['Admission Rate (%)'] = (course_adm['Admitted'] / course_adm['Total'] * 100).round(1)
course_adm = course_adm.sort_values('Admission Rate (%)', ascending=True)
fig = px.bar(course_adm, x='Admission Rate (%)', y='Course', orientation='h',
             color='Admission Rate (%)',
             color_continuous_scale=['#BFDBFE', PALETTE['primary']],
             text='Admission Rate (%)')
fig.update_traces(texttemplate='%{text}%', textposition='outside', marker_line_width=0)
fig.update_coloraxes(showscale=False)
clean_layout(fig, 'Admission Rate (%) by Course', height=420)
fig.show()

In [ ]:
counsellor = admissions.groupby('attended_by')['is_admitted'].agg(['sum', 'count']).reset_index()
counsellor.columns = ['Counsellor', 'Admitted', 'Total Handled']
counsellor['Not Admitted'] = counsellor['Total Handled'] - counsellor['Admitted']
fig = go.Figure()
fig.add_trace(go.Bar(name='Admitted', x=counsellor['Counsellor'], y=counsellor['Admitted'],
                     marker_color=PALETTE['secondary'], text=counsellor['Admitted'],
                     textposition='inside', marker_line_width=0))
fig.add_trace(go.Bar(name='Not Admitted', x=counsellor['Counsellor'], y=counsellor['Not Admitted'],
                     marker_color='#FCA5A5', text=counsellor['Not Admitted'],
                     textposition='inside', marker_line_width=0))
fig.update_layout(barmode='stack')
clean_layout(fig, 'Counsellor Performance — Admissions Handled')
fig.show()

## 🔗 6. Combined Insights

In [ ]:
inq_course = admissions.groupby('course').size().reset_index(name='Inquiries')
enr_course = students.groupby('course').size().reset_index(name='Enrolled')
funnel_df  = pd.merge(inq_course, enr_course, on='course', how='outer').fillna(0)
funnel_df  = funnel_df.sort_values('Inquiries', ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(name='Inquiries', x=funnel_df['course'], y=funnel_df['Inquiries'],
                     marker_color='#BFDBFE', marker_line_width=0))
fig.add_trace(go.Bar(name='Enrolled',  x=funnel_df['course'], y=funnel_df['Enrolled'],
                     marker_color=PALETTE['primary'], marker_line_width=0))
fig.update_layout(barmode='group')
clean_layout(fig, 'Inquiry vs Enrollment by Course')
fig.update_xaxes(tickangle=-20)
fig.show()

In [ ]:
monthly_adm = (admissions.groupby('admission_month').size()
               .reset_index(name='Inquiries').sort_values('admission_month'))
monthly_stu = (students.groupby('enrollment_month').size()
               .reset_index(name='Enrolled')
               .rename(columns={'enrollment_month': 'admission_month'})
               .sort_values('admission_month'))
combined = pd.merge(monthly_adm, monthly_stu, on='admission_month', how='outer').fillna(0)
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(name='Inquiries', x=combined['admission_month'], y=combined['Inquiries'],
                     marker_color='#BFDBFE', marker_line_width=0), secondary_y=False)
fig.add_trace(go.Scatter(name='Enrolled', x=combined['admission_month'], y=combined['Enrolled'],
                         mode='lines+markers',
                         line=dict(color=PALETTE['primary'], width=2.5),
                         marker=dict(size=6)), secondary_y=True)
fig.update_layout(height=420, margin=dict(l=20,r=20,t=55,b=20),
                  paper_bgcolor='white', plot_bgcolor='white',
                  font=dict(family='Arial', size=12),
                  title=dict(text='Monthly Inquiries vs Enrolled (Dual Axis)', font=dict(size=15), x=0),
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                              bgcolor='rgba(0,0,0,0)', font=dict(size=11)),
                  xaxis=dict(showgrid=False, linecolor='#E2E8F0', tickangle=-30))
fig.update_yaxes(title_text='Inquiries', secondary_y=False, gridcolor='#F3F4F6')
fig.update_yaxes(title_text='Enrolled',  secondary_y=True)
fig.show()

In [ ]:
qual_counts = admissions['qualification'].value_counts().reset_index()
qual_counts.columns = ['Qualification', 'Count']
fig = px.bar(qual_counts, x='Qualification', y='Count', color='Qualification',
             color_discrete_sequence=PALETTE['courses'], text='Count')
fig.update_traces(marker_line_width=0, textposition='outside')
fig.update_layout(showlegend=False)
clean_layout(fig, 'Inquiry Leads by Qualification')
fig.show()

In [ ]:
dur_counts = students['duration'].value_counts().reset_index()
dur_counts.columns = ['Duration', 'Students']
fig = px.pie(dur_counts, names='Duration', values='Students', hole=0.5,
             color_discrete_sequence=[PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']])
fig.update_traces(textposition='outside', textinfo='percent+label',
                  marker=dict(line=dict(color='white', width=2)))
clean_layout(fig, 'Course Duration Preference')
fig.show()

## 🗂️ 7. Raw Data Explorer

In [ ]:
print('=== Students Dataset ===')
display(students.reset_index(drop=True))

In [ ]:
print('=== Admissions Dataset ===')
display(admissions.reset_index(drop=True))